<a href="https://colab.research.google.com/github/Kaluvai1203/CSA6102---Lab-/blob/main/EXP-14%20.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
!pip install Pillow

In [16]:
import re
from urllib.parse import urlparse


sample_email = """From: PayPal Support <support@paypa1.com>
Return-Path: attacker@gmail.com
Subject: Urgent! Verify Your Account

Dear Customer,

Your PayPal account has been suspended.

Click the link below immediately to verify your account:

http://paypa1-login-security.com/login

You may also visit:
http://192.168.100.50/paypal

Failure to verify within 24 hours will permanently suspend your account.

Regards,
PayPal Team
"""

with open("phishing_email.txt", "w") as f:
    f.write(sample_email)

print("Sample phishing email created successfully.\n")


filename = input("Enter email file name: ")

with open(filename, "r") as file:
    email = file.read()


from_match = re.search(r"From:\s.*<(.+?)>", email)
return_match = re.search(r"Return-Path:\s(.+)", email)

from_email = from_match.group(1) if from_match else "Unknown"
return_email = return_match.group(1) if return_match else "Unknown"

print("\n========== HEADER ANALYSIS ==========")
print("From Address :", from_email)
print("Return-Path  :", return_email)


risk_score = 0

from_domain = from_email.split("@")[-1]
return_domain = return_email.split("@")[-1]

if from_domain != return_domain:
    print("\nSender Verification : FAILED")
    print("Spoofed sender detected.")
    risk_score += 30
else:
    print("\nSender Verification : PASSED")


urls = re.findall(r'https?://[^\s]+', email)

print("\n========== URL ANALYSIS ==========")

suspicious_keywords = [
    "login",
    "verify",
    "secure",
    "update",
    "bank",
    "paypal",
    "account"
]

ioc_domains = []

for url in urls:

    domain = urlparse(url).netloc
    ioc_domains.append(domain)

    suspicious = False

    if any(word in domain.lower() for word in suspicious_keywords):
        suspicious = True

    if re.search(r"\d+\.\d+\.\d+\.\d+", domain):
        suspicious = True

    print("\nURL :", url)
    print("Domain :", domain)

    if suspicious:
        print("Status : Suspicious")
        risk_score += 20
    else:
        print("Status : Legitimate")


if risk_score >= 70:
    level = "HIGH"
elif risk_score >= 40:
    level = "MEDIUM"
else:
    level = "LOW"

print("\n========== PHISHING REPORT ==========")

print("Risk Score :", risk_score)
print("Risk Level :", level)

print("\nIndicators of Compromise (IoCs)")
for d in ioc_domains:
    print("-", d)

print("\nSummary Statistics")
print("------------------")
print("Total URLs :", len(urls))
print("Suspicious URLs :", risk_score // 20)
print("Spoofed Sender :", "Yes" if from_domain != return_domain else "No")

print("\nConclusion:")
if level == "HIGH":
    print("This email is highly likely to be a phishing email.")
elif level == "MEDIUM":
    print("This email contains several phishing indicators.")
else:
    print("No major phishing indicators detected.")

Sample phishing email created successfully.

Enter email file name: phishing_email.txt

========== HEADER ANALYSIS ==========
From Address : support@paypa1.com
Return-Path  : attacker@gmail.com

Sender Verification : FAILED
Spoofed sender detected.

========== URL ANALYSIS ==========

URL : http://paypa1-login-security.com/login
Domain : paypa1-login-security.com
Status : Suspicious

URL : http://192.168.100.50/paypal
Domain : 192.168.100.50
Status : Suspicious

========== PHISHING REPORT ==========
Risk Score : 70
Risk Level : HIGH

Indicators of Compromise (IoCs)
- paypa1-login-security.com
- 192.168.100.50

Summary Statistics
------------------
Total URLs : 2
Suspicious URLs : 3
Spoofed Sender : Yes

Conclusion:
This email is highly likely to be a phishing email.
